# Pixel-Art Diffusion — T4 Training

Runs the pixel-space DDPM from [`Bit-Sahil04/toy-pixel-diffuser`](https://github.com/Bit-Sahil04/toy-pixel-diffuser) on a Colab **T4 (16GB)**.

T4 settings vs the local 4GB defaults: **batch_size 64, grad_accum 1** (effective batch 64), `--max_vram_gb 16`. The model architecture is unchanged.

**Runtime → Change runtime type → T4 GPU** before running.


In [ ]:
# 1. Confirm the GPU

!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader


## 2. Get the code + dependencies

Colab already has a CUDA build of torch — we only install the missing packages (do **not** reinstall torch here; `requirements.txt`'s cu130 pin is for the local Windows machine).


In [ ]:
!git clone https://github.com/Bit-Sahil04/toy-pixel-diffuser.git

%cd toy-pixel-diffuser

!pip install -q datasets huggingface_hub pillow numpy tqdm matplotlib


In [ ]:
# 3. Environment sanity check (CUDA + AMP on the T4)

!python check_env.py


## 4. (Recommended) Mount Google Drive for checkpoints

Colab disks are wiped when the runtime dies. This saves checkpoints/sample grids/logs to Drive so a disconnected session loses at most one checkpoint interval.

Skip this cell to train fully ephemeral.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

DRIVE_CKPT = '/content/drive/MyDrive/pixel_diffuser/run1'

DRIVE_SAMPLES = '/content/drive/MyDrive/pixel_diffuser/run1_samples'

DRIVE_LOG = '/content/drive/MyDrive/pixel_diffuser/run1_log.csv'


## 5. Download + inspect the dataset

~300MB (LPC 4-view sprites, 50k images @ 128x128 RGBA). Prints real dims/modes and a sample caption; asserts no flip/rotation transforms.


In [ ]:
!python data.py


## 6. (Optional) Quick smoke test first

60 steps on 512 images to confirm the T4 setup end-to-end before the real run (~2 min, most of it the fixed-seed sample grid).


In [ ]:
!python -u train.py --limit 512 --max_train_steps 60 --batch_size 64 --grad_accum_steps 1 --max_vram_gb 16


## 7. Train

Defaults: 20k steps, batch 64, checkpoint every 500 steps, fixed-seed 4x4 grid every 200 steps (see `config.py`).

**Interrupted?** Just re-run this cell with the resume line below — optimizer/EMA/AMP state restores exactly.


In [ ]:
# fresh run:

!python -u train.py --batch_size 64 --grad_accum_steps 1 --max_vram_gb 16 \n
    --checkpoint_dir "$DRIVE_CKPT" --samples_dir "$DRIVE_SAMPLES" --log_csv "$DRIVE_LOG"



# ...or resume after an interruption:

# !python -u train.py --batch_size 64 --grad_accum_steps 1 --max_vram_gb 16 \n
#     --resume_from "$DRIVE_CKPT/latest.pt" \n
#     --checkpoint_dir "$DRIVE_CKPT" --samples_dir "$DRIVE_SAMPLES" --log_csv "$DRIVE_LOG"


## 8. Watch progress

Sample grids use a FIXED seed: every frame denoises the same noise, so you literally watch sprites emerge.


In [ ]:
!python make_gif.py --samples_dir "$DRIVE_SAMPLES"
from IPython.display import Image
Image(filename=f'{DRIVE_SAMPLES}/training_progress.gif')

## 9. Inference from the latest checkpoint

Uses EMA weights; 16 sprites via the full 1000-step DDPM sampler (~4 min on T4 at batch 16).


In [ ]:
!python sample.py --checkpoint "$DRIVE_CKPT/latest.pt" --n 16 --nrow 4 --save_individual
from IPython.display import Image
import glob
Image(filename=sorted(glob.glob('samples/infer_step*.png'))[-1])

## Notes

- If batch 64 ever OOMs (shouldn't on 16GB — heuristic estimate ~11GB), drop to `--batch_size 32 --grad_accum_steps 2`.

- Loss CSV lives at the `$DRIVE_LOG` path; plot with `pandas` if you want curves.

- Colab free tier disconnects after idle timeouts — the resume line in cell 7 makes that a non-event.
